##**[전복 나이 분류 모델] 딥러닝 기반 Classification**

전복의 성별('M', 'F', 'I')을 예측

> 성별 분류는 특히 '유체(Infant)'와 '성체(Male/Female)' 간의 물리적 차이를 잘 구분하느냐가 핵심


In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras import layers, models

In [3]:
# 1. 데이터 로드 (abalone)
path_abalone = "/content/drive/MyDrive/가천대학교/2026-1/인공지능개론/Colab Notebooks/2. 데이터/abalone.csv"
df = pd.read_csv(path_abalone, index_col=0)
df

,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight,Rings
id,,,,,,,,,
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.1500,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.0700,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.2100,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.1550,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.0550,7
...,...,...,...,...,...,...,...,...,...
4172,F,0.565,0.450,0.165,0.8870,0.3700,0.2390,0.2490,11
4173,M,0.590,0.440,0.135,0.9660,0.4390,0.2145,0.2605,10
4174,M,0.600,0.475,0.205,1.1760,0.5255,0.2875,0.3080,9


In [5]:
# [단계 1] 예측에 사용할 입력 데이터(X)와 맞출 정답(y) 분리하기
y = df['Sex']                        # 예측하려는 목표인 '성별'만 따로 저장
X = df.drop('Sex', axis=1)           # 성별을 제외한 나머지 신체 데이터만 저장

# [단계 2] 데이터 간의 단위를 맞춰주는 표준화(스케일링) 작업
scaler = StandardScaler()
X = scaler.fit_transform(X)          # 평균 0, 표준편차 1이 되도록 데이터 크기 조정

# [단계 3] 모델 학습용 데이터와 검증(시험)용 데이터 나누기
# (75%는 학습에 사용하고, 25%는 시험용으로 따로 빼둠)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

# 딥러닝 모델은 숫자로 된 행렬 계산만 할 수 있으므로, 문자를 숫자로 바꾸는 과정이 필수적

문자를 숫자로 바꿀 때, 단순히 F = 0, I = 1, M = 2와 같이 연속된 정수로만 바꾸면(레이블 인코딩)
- "2(M)는 0(F)보다 2배 더 크구나."
- "0(F)과 2(M)의 평균은 1(I)이구나."

>성별은 크고 작음이 없는 독립적인 범주(Category)입니다. 이를 원-핫 인코딩을 통해 [1, 0, 0], [0, 1, 0], [0, 0, 1]과 같은 벡터로 만들어 주어야 모든 성별이 동등한 관계(거리가 동일한 관계)가 된다

In [10]:
# [단계 4] 정답(성별) 데이터 원-핫 인코딩 (문자를 0과 1로 변환)
model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(y_train.shape[1], activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=16,
    verbose=1
)

Epoch 1/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.4982 - loss: 0.9451 - val_accuracy: 0.5502 - val_loss: 0.8523
Epoch 2/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5397 - loss: 0.8867 - val_accuracy: 0.5694 - val_loss: 0.8346
Epoch 3/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5345 - loss: 0.8787 - val_accuracy: 0.5694 - val_loss: 0.8271
Epoch 4/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5393 - loss: 0.8670 - val_accuracy: 0.5821 - val_loss: 0.8206
Epoch 5/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5485 - loss: 0.8624 - val_accuracy: 0.5869 - val_loss: 0.8159
Epoch 6/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5565 - loss: 0.8669 - val_accuracy: 0.5917 - val_loss: 0.8159
Epoch 7/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5485 - loss: 0.8597 - val_accuracy: 0.6061 - val_loss: 0.8143
Epoch 8/50
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5517 - loss: 0.8565 - val_accuracy: 0.

In [11]:
# [단계 5] 평가
y_pred = model.predict(X_test)

y_test_class = np.argmax(y_test, axis=1)
y_pred_class = np.argmax(y_pred, axis=1)

print(classification_report(y_test_class, y_pred_class))
print(confusion_matrix(y_test_class, y_pred_class))

33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
              precision    recall  f1-score   support

           0       0.47      0.46      0.47       316
           1       0.72      0.78      0.75       359
           2       0.50      0.47      0.49       370

    accuracy                           0.57      1045
   macro avg       0.56      0.57      0.57      1045
weighted avg       0.57      0.57      0.57      1045

[[145  41 130]
 [ 35 281  43]
 [127  69 174]]
